Knowledge Engineering: Self-Driving Trailer as a knowledge-Based Agent

Step 1 : Identify the task

Task : Develop a knowledge-based self-driving trailer agent that can navigate between towns using a road network, select the lowest-cost available route and make decisions based on stored knowledge and current road conditions.

Goal : Enable the trailer to travel safely and efficiently from its initial location to a specified destination while minimizing total travel distance and avoiding blocked roads.

Step 2 : Assemble relevant knowledge

Information from Exercise 3 will be reused. For instance the directed graph and average road distances.

No external APIs or web scraping tool were used in this implementation.

Step 3 : Define vocabulary

------------------------------------------------------------------------------------------
VOCABULARY                          MEANING
------------------------------------------------------------------------------------------
Town(x)                             Identifies a town in the road network
Connected(x,y)                      Indicates a directed road from town x to town y
Distance(x,y,d)                     Represents the distance between two connected towns
Blocked(x,y)                        Indicates that a road is unavailable
CurrentLocation(x)                  Represents the trailer's current location
Destination(x)                      Represents the intended destination
Route(x,y)                          Represents a possible route between two towns
Action(a)                           Represents an action the trailer can perform
-----------------------------------------------------------------------------------------

Constants :
Towns : Nairobi, Naivasha, Nakuru, Eldoret, Kitale, Kakamega, Kisumu, Kericho, Sondu, Narok, Bomet, Kisii, Thika, Nyeri, and Nyahururu

Road Conditions : Clear, Blocked

Actions : Continue, Stop, Reroute, Arrived

Step 4 : Encode general Knowledge about the domain

Things the agent needs to understand :
1. A trailer can travel from one town to another only if a directed road exists
2. Every road has a distance which contributes to the total route cost
3. A blocked road cannot be used
4. The trailer should select the available route with the lowest total distance
5. If a road becomes blocked, the agent should search for an alternative route
6. If no route is available, the agent should stop and report that no route was found
7. If the trailer reaches its destination, it should stop navigating

In [1]:
# Knowledge Base: Road network and distances

GRAPH = {
    "Nairobi":   {"Naivasha": 89.5, "Narok": 141.5, "Thika": 44},
    "Naivasha":  {"Nakuru": 71},
    "Nakuru":    {"Eldoret": 156, "Kericho": 107},
    "Eldoret":   {"Kitale": 71},
    "Kitale":    {"Kakamega": 110},
    "Kakamega":  {"Kisumu": 50},
    "Kericho":   {"Sondu": 45, "Kisii": 87.2},
    "Sondu":     {"Kisumu": 52.6},
    "Narok":     {"Bomet": 81.5},
    "Bomet":     {"Kericho": 74.25},
    "Kisii":     {"Sondu": 61.3},
    "Thika":     {"Nyeri": 110},
    "Nyeri":     {"Nyahururu": 104},
    "Nyahururu": {"Nakuru": 64},
    "Kisumu":    {}
}

# Store road conditions separately
BLOCKED_ROADS = set()

# General knowledge rules
RULES = {
    "road_exists": "The trailer can travel only along an existing directed road.",
    "road_blocked": "The trailer must not use a blocked road.",
    "shortest_route": "Select the available route with the lowest total distance.",
    "alternative_route": "If a road is blocked, search for another available route.",
    "no_route": "If no route exists, stop and report that no route is available.",
    "destination_reached": "If the destination is reached, stop navigation."
}

# Display the knowledge base
print("GENERAL KNOWLEDGE RULES")

for rule, description in RULES.items():
    print(f"{rule}: {description}")

print("\nRoad network loaded successfully.")
print("Number of towns:", len(GRAPH))

GENERAL KNOWLEDGE RULES
road_exists: The trailer can travel only along an existing directed road.
road_blocked: The trailer must not use a blocked road.
shortest_route: Select the available route with the lowest total distance.
alternative_route: If a road is blocked, search for another available route.
no_route: If no route exists, stop and report that no route is available.
destination_reached: If the destination is reached, stop navigation.

Road network loaded successfully.
Number of towns: 15


Step 5 : Add new facts to the knowledge base

A knowledge-based agent should be able to acquire new information and update its knowledge.

Functions will be implemented to : 
- Display the current knowledge base
- Add a new toad and its distance
- Mark an existing road as blocked
- Restore a road when it becomes available again


In [3]:
# Display the current road network
def show_kb():
    print("\n   CURRENT KNOWLEDGE BASE   ")

    for town, connections in GRAPH.items():
        for neighbour, distance in connections.items():

            if (town, neighbour) in BLOCKED_ROADS:
                status = "BLOCKED"
            else:
                status = "CLEAR"

            print(f"{town} -> {neighbour}: {distance} km [{status}]")



# Add a new road to the knowledge base
def add_road(start, end, distance):

    if distance <= 0:
        print("Invalid distance. Distance must be positive.")
        return

    GRAPH.setdefault(start, {})
    GRAPH.setdefault(end, {})

    GRAPH[start][end] = distance

    print(f"Added road: {start} -> {end} ({distance} km)")


# Mark a road as blocked
def block_road(start, end):

    if end not in GRAPH.get(start, {}):
        print("Road does not exist.")
        return

    BLOCKED_ROADS.add((start, end))

    print(f"Road blocked: {start} -> {end}")


# Restore a blocked road
def unblock_road(start, end):

    if (start, end) in BLOCKED_ROADS:
        BLOCKED_ROADS.remove((start, end))
        print(f"Road restored: {start} -> {end}")
    else:
        print("This road is not currently blocked.")


# Example: add a new fact about a blocked road
block_road("Kericho", "Sondu")

# Display the updated knowledge base
show_kb()

Road blocked: Kericho -> Sondu

   CURRENT KNOWLEDGE BASE   
Nairobi -> Naivasha: 89.5 km [CLEAR]
Nairobi -> Narok: 141.5 km [CLEAR]
Nairobi -> Thika: 44 km [CLEAR]
Naivasha -> Nakuru: 71 km [CLEAR]
Nakuru -> Eldoret: 156 km [CLEAR]
Nakuru -> Kericho: 107 km [CLEAR]
Eldoret -> Kitale: 71 km [CLEAR]
Kitale -> Kakamega: 110 km [CLEAR]
Kakamega -> Kisumu: 50 km [CLEAR]
Kericho -> Sondu: 45 km [BLOCKED]
Kericho -> Kisii: 87.2 km [CLEAR]
Sondu -> Kisumu: 52.6 km [CLEAR]
Narok -> Bomet: 81.5 km [CLEAR]
Bomet -> Kericho: 74.25 km [CLEAR]
Kisii -> Sondu: 61.3 km [CLEAR]
Thika -> Nyeri: 110 km [CLEAR]
Nyeri -> Nyahururu: 104 km [CLEAR]
Nyahururu -> Nakuru: 64 km [CLEAR]


Explanation : 
The block_road() function adds a road to the BLOCKED_ROADS set. This represents new information received by the agent.

The unblock_road() function removes the road from the blocked list when it becomes available again.

The add_road() function allows the knowledge base to be extended with new road information.

Step 6 : Encode the specific problem instance

The agent now needs to receive information about the current navigation task.

We define:
- The trailer's initial location
- The destination
- The current road conditions

he agent begins in Nairobi and must reach Kisumu. The blocked road information is also included so that the agent can consider it when planning its journey.

In [4]:
# Current navigation facts

INITIAL_STATE = "Nairobi"
GOAL_STATE = "Kisumu"

# Current location of the trailer
current_location = INITIAL_STATE

# Display the problem instance
print("   CURRENT PROBLEM INSTANCE   ")
print("Current location:", current_location)
print("Destination:", GOAL_STATE)
print("Blocked roads:", BLOCKED_ROADS)

   CURRENT PROBLEM INSTANCE   
Current location: Nairobi
Destination: Kisumu
Blocked roads: {('Kericho', 'Sondu')}


Step 7 : Pose queries and get answers

The agent needs a way to reason about the stored knowledge and answer navigation questions.

Uniform Cost Search (UCS) allows the search algorithm to find the lowest-cost route using only available roads.

The algorithm uses a priority queue to expand the town with the lowest accumulated distance first.

When the destination is reached, it returns the optimal path and its total distance.

If no route is available, it returns None and infinity.

In [5]:
import heapq

# Check whether the destination has been reached
def goal_test(state, goal):
    return state == goal


# Uniform Cost Search using the knowledge base
def uniform_cost_search(graph, start, goal):

    # Check whether the towns exist
    if start not in graph or goal not in graph:
        return None, float("inf")

    counter = 0

    # Priority queue: (path cost, tie-breaker, town, path)
    frontier = [(0, counter, start, [start])]

    # Store the cheapest known cost to each town
    best_cost = {start: 0}

    explored = set()

    while frontier:

        cost, _, town, path = heapq.heappop(frontier)

        # Ignore towns already expanded
        if town in explored:
            continue

        # Check whether the destination has been reached
        if goal_test(town, goal):
            return path, cost

        explored.add(town)

        # Examine connected towns
        for neighbour, distance in graph[town].items():

            # Ignore blocked roads
            if (town, neighbour) in BLOCKED_ROADS:
                continue

            new_cost = cost + distance

            # Update if a cheaper route is found
            if new_cost < best_cost.get(neighbour, float("inf")):

                best_cost[neighbour] = new_cost

                counter += 1

                heapq.heappush(
                    frontier,
                    (new_cost, counter, neighbour, path + [neighbour])
                )

    # No available route was found
    return None, float("inf")

Step 8 : Query the knowledge base and make decisions

The agent should be able to  :
- Find the optimal route
- Identify the next town to travel to
- Determine whether to continue or reroute
- Stop if the destination is reached
- Report when no available route exists

In [7]:
# Query the knowledge base for available roads
def roads_from(town):

    if town not in GRAPH:
        return "Unknown town."

    return GRAPH[town]


# Make a navigation decision
def trailer_agent(current, destination):

    # Rule 1: Check if destination has been reached
    if current == destination:
        return "ARRIVED: Destination reached. Stop navigation."

    # Rule 2: Find the best available route
    path, cost = uniform_cost_search(
        GRAPH, current, destination
    )

    # Rule 3: Check if a route exists
    if path is None:
        return "STOP: No available route to the destination."

    # Identify the next town
    next_town = path[1]

    # Rule 4: Decide whether to continue or reroute
    if BLOCKED_ROADS:
        action = "REROUTE"
    else:
        action = "CONTINUE"

    return {
        "Decision": action,
        "Current Location": current,
        "Next Town": next_town,
        "Destination": destination,
        "Route": path,
        "Total Distance": round(cost, 2)
    }


# Ask the agent to plan the journey
decision = trailer_agent(current_location, GOAL_STATE)

print("\nTRAILER AGENT DECISION   ")

if isinstance(decision, dict):

    for key, value in decision.items():
        print(f"{key}: {value}")

else:
    print(decision)


TRAILER AGENT DECISION   
Decision: REROUTE
Current Location: Nairobi
Next Town: Naivasha
Destination: Kisumu
Route: ['Nairobi', 'Naivasha', 'Nakuru', 'Kericho', 'Kisii', 'Sondu', 'Kisumu']
Total Distance: 468.6


Explanation : 

The trailer_agent() function acts as the decision-making component.

It first checks whether the trailer has reached its destination. If not it uses UCS to find an available route.

If a route exists, it identifies the next town and returns the recommended action and route details.

In this demonstration, Kericho to Sondu is blocked. Therefore, the agent should find an alternative route instead of following the original route.

Querying Individual Facts

The knowledge base can be asked specific questions.

These queries demonstrate how the agent retrieves information from its knowledge base.

The agent can retrieve direct road connections, check distances, determine road conditions, and apply its rules to make decisions.

In [8]:
# Query 1: Which towns can be reached directly from Nairobi?
print("Roads from Nairobi:", roads_from("Nairobi"))

# Query 2: What is the distance from Nairobi to Thika?
print("Distance to Thika:", GRAPH["Nairobi"]["Thika"], "km")

# Query 3: Is the road from Kericho to Sondu blocked?
print("Kericho to Sondu blocked:",
      ("Kericho", "Sondu") in BLOCKED_ROADS)

# Query 4: What should the agent do when it reaches Kisumu?
print("Decision at destination:",
      trailer_agent("Kisumu", "Kisumu"))

Roads from Nairobi: {'Naivasha': 89.5, 'Narok': 141.5, 'Thika': 44}
Distance to Thika: 44 km
Kericho to Sondu blocked: True
Decision at destination: ARRIVED: Destination reached. Stop navigation.


Step 9 : Debugging the knowledge base

Debugging allows us to check whether the agent responds correctly to different situations including unexpected or invalid information.

The following cases will be tested :
1. A normal navigation request
2. A blocked road
3. An unknown town
4. A destination that cannot be reached
5. A destination that has already been reached

In [9]:
print("\nDEBUGGING THE TRAILER AGENT")

# Test 1: Normal navigation with a blocked road
print("\nTest 1: Find an alternative route")
print(trailer_agent("Nairobi", "Kisumu"))


# Test 2: Check an unknown town
print("\nTest 2: Unknown town")
print("Roads from Atlantis:", roads_from("Atlantis"))


# Test 3: Check an unknown destination
print("\nTest 3: Unknown destination")
print(trailer_agent("Nairobi", "Atlantis"))


# Test 4: Destination already reached
print("\nTest 4: Destination reached")
print(trailer_agent("Kisumu", "Kisumu"))


# Test 5: No available route
print("\nTest 5: Block all roads leaving Nairobi")

# Temporarily save the existing blocked roads
previous_blocked = BLOCKED_ROADS.copy()

# Temporarily block every road leaving Nairobi
for neighbour in GRAPH["Nairobi"]:
    BLOCKED_ROADS.add(("Nairobi", neighbour))

print(trailer_agent("Nairobi", "Kisumu"))

# Restore the original road conditions
BLOCKED_ROADS.clear()
BLOCKED_ROADS.update(previous_blocked)

print("\nOriginal road conditions restored.")


DEBUGGING THE TRAILER AGENT

Test 1: Find an alternative route
{'Decision': 'REROUTE', 'Current Location': 'Nairobi', 'Next Town': 'Naivasha', 'Destination': 'Kisumu', 'Route': ['Nairobi', 'Naivasha', 'Nakuru', 'Kericho', 'Kisii', 'Sondu', 'Kisumu'], 'Total Distance': 468.6}

Test 2: Unknown town
Roads from Atlantis: Unknown town.

Test 3: Unknown destination
STOP: No available route to the destination.

Test 4: Destination reached
ARRIVED: Destination reached. Stop navigation.

Test 5: Block all roads leaving Nairobi
STOP: No available route to the destination.

Original road conditions restored.


Explanation : 

-------------------------------------------------------------------------
TEST                            EXPECTED BEHAIVOUR
-------------------------------------------------------------------------
Normal navigation	            Finds an available route
Unknown town	                Reports that the town is unknown
Unknown destination	            Reports that no route is available
Destination reached	            Tells the trailer to stop navigation
All roads from Nairobi blocked	Reports that no available route exists
------------------------------------------------------------------------

The last test temporarily blocks all roads leaving Nairobi. After the test, the original road conditions are restored so that the knowledge base is not permanently changed.

Step 10 : Final Reflection

The development of the self-driving trailer as a knowledge-based agent demonstrated how artificial intelligence can use stored facts and rules to make navigation decisions. The agent used a directed, weighted graph to represent the road network and Uniform Cost Search to identify the route with the lowest total distance.

One of the main challenges was ensuring that the agent could respond to changes in road conditions. This was addressed by introducing a knowledge base that stores blocked roads separately from the original graph. As a result, the agent can avoid unavailable roads and search for alternative routes.

Another challenge was handling unexpected situations such as unknown towns and destinations with no available routes. Debugging functions and test cases were introduced to ensure that the agent responds appropriately instead of producing incorrect navigation instructions.

However, the current implementation has some limitations. It uses predefined distances and manually updated road conditions rather than real-time traffic information. It also considers distance as the main optimization factor and does not account for travel time, fuel consumption, weather,or road safety conditions.

Future improvements could include integrating GPS data, traffic APIs, road-closure information and sensors to provide real-time updates. Additional search algorithms and distance measures could also be used to compare route efficiency.

Overall, the exercise demonstrates how a knowledge-based agent can combine a knowledge base, inference rules and a search algorithm to make informed navigation decisions.